In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import torch

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:

train_root = '/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train'  # adjust to your path
folders = sorted(os.listdir(train_root), key=int)
print(folders[:12])

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']


In [6]:
import torch
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname, f'({len(filenames)} files)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


/kaggle/input (0 files)
/kaggle/input/competitions (0 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project (1 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/test (1036 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train (0 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train/7 (10 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train/47 (9 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train/17 (10 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train/81 (11 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train/19 (10 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train/22 (10 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train/2 (10 files)
/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project/train/35 (5 files)
/kaggle/input

In [8]:
DATA_ROOT  = '/kaggle/input/competitions/ucsc-cse-144-spring-2026-final-project'
TRAIN_DIR  = os.path.join(DATA_ROOT, 'train')
TEST_DIR   = os.path.join(DATA_ROOT, 'test')

folders = sorted(os.listdir(TRAIN_DIR), key=int)
print('First 12 class folders in order:', folders[:12])
print('Total classes:', len(folders))

First 12 class folders in order: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']
Total classes: 100


In [9]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ImageNet normalization stats — required because our model was pretrained on ImageNet
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Training transforms: heavy augmentation because we have only 10 images per class.
# Each epoch the model sees randomly cropped/flipped/jittered versions, which
# artificially expands our tiny dataset and fights overfitting.
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25),
])

# Test transforms: NO randomness. Just resize + center crop, so predictions are stable.
test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class TrainDataset(Dataset):
    def __init__(self, root, transform):
        self.transform = transform
        self.samples = []
        for cls in sorted(os.listdir(root), key=int):   # key=int again — critical
            label = int(cls)                              # folder "7" -> label 7
            cls_dir = os.path.join(root, cls)
            for fname in os.listdir(cls_dir):
                self.samples.append((os.path.join(cls_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')  # convert handles grayscale/CMYK images
        return self.transform(img), label

full_ds = TrainDataset(TRAIN_DIR, train_tf)
print('Total training images:', len(full_ds))   # expect ~1000

Total training images: 1079


In [10]:
# Hold out a small validation set so we can watch for overfitting.
# We can't trust the public leaderboard (it's only ~10% of test data), so we
# need our own honest accuracy estimate.
from torch.utils.data import random_split

val_size = 100                      # ~1 image per class
train_size = len(full_ds) - val_size
g = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(full_ds, [train_size, val_size], generator=g)

# Validation should NOT use random augmentation, so give it the clean transforms.
# (Small hack: wrap it so it uses test_tf.)
class ApplyTF(Dataset):
    def __init__(self, subset, tf):
        self.subset, self.tf = subset, tf
    def __len__(self): return len(self.subset)
    def __getitem__(self, i):
        path, label = self.subset.dataset.samples[self.subset.indices[i]]
        return self.tf(Image.open(path).convert('RGB')), label

val_ds = ApplyTF(val_ds, test_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
print('Train:', len(train_ds), 'Val:', len(val_ds))

Train: 979 Val: 100


In [12]:
import torch.nn as nn
from torchvision import models

# ConvNeXt-Tiny: strong, modern, but small enough to not instantly overfit 1000 images.
# weights=...IMAGENET1K_V1 downloads weights pretrained on 1.2M ImageNet images.
# Transfer learning = reuse those learned features, just teach a new final layer.
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

# Replace the final classification layer: ImageNet had 1000 classes, we have 100.
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, 100)
model = model.to(device)
print('Model ready.')

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 213MB/s] 


Model ready.


In [ ]:
import torch.nn.functional as F

EPOCHS = 50
# label_smoothing makes the model less overconfident — helps a lot on tiny datasets
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

best_acc = 0
for epoch in range(EPOCHS):
    model.train()
    running = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item()
    scheduler.step()
    val_acc = evaluate(val_loader)
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')  # keep the best version
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | loss {running/len(train_loader):.3f} | val_acc {val_acc:.3f}')

print('Best validation accuracy:', best_acc)

Epoch  1/50 | loss 0.995 | val_acc 0.690
Epoch  2/50 | loss 0.952 | val_acc 0.720
Epoch  3/50 | loss 0.939 | val_acc 0.710
Epoch  4/50 | loss 0.945 | val_acc 0.760
Epoch  5/50 | loss 0.922 | val_acc 0.750
Epoch  6/50 | loss 0.912 | val_acc 0.710
Epoch  7/50 | loss 0.902 | val_acc 0.710
Epoch  8/50 | loss 0.901 | val_acc 0.740
Epoch  9/50 | loss 0.890 | val_acc 0.780
Epoch 10/50 | loss 0.901 | val_acc 0.720
Epoch 11/50 | loss 0.875 | val_acc 0.680


In [14]:
import pandas as pd
import re

# Load the best saved weights
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Test images are named 0.jpg ... 999.jpg. We must predict each by its numeric ID.
test_files = sorted(os.listdir(TEST_DIR), key=lambda f: int(re.findall(r'\d+', f)[0]))

rows = []
with torch.no_grad():
    for fname in test_files:
        img_id = int(re.findall(r'\d+', fname)[0])
        img = Image.open(os.path.join(TEST_DIR, fname)).convert('RGB')
        x = test_tf(img).unsqueeze(0).to(device)   # unsqueeze adds the batch dimension
        pred = model(x).argmax(1).item()
        rows.append((img_id, pred))

sub = pd.DataFrame(rows, columns=['ID', 'Label']).sort_values('ID')
sub.to_csv('submission.csv', index=False)
print(sub.head())
print('Wrote submission.csv with', len(sub), 'rows')   # expect 1000

   ID  Label
0   0     62
1   1     43
2   2     38
3   3     62
4   4     42
Wrote submission.csv with 1036 rows
